In [1]:
# 0. Environment Setup

# Clone the new kauldron repository
!git clone https://github.com/google-research/kauldron || true
!pip install -q ./kauldron
# Clone the gemma repository if not already present
!git clone https://github.com/google-deepmind/gemma.git || true
!pip install -q ./gemma

# Clone the dialog repository to fix Gemma format issues
!git clone https://github.com/google-deepmind/dialog || true
!pip install -q ./dialog

!pip install -q flax optax seqio



# Ensure our local modules are in the Python path
import sys
import os
sys.path.append(os.getcwd())

Cloning into 'kauldron'...
remote: Enumerating objects: 9546, done.
remote: Counting objects: 100% (162/162), done.
remote: Compressing objects: 100% (134/134), done.
remote: Total 9546 (delta 49), reused 38 (delta 28), pack-reused 9384 (from 3)
Receiving objects: 100% (9546/9546), 2.83 MiB | 724.00 KiB/s, done.
Resolving deltas: 100% (6857/6857), done.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 89.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.8/101.8 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 

In [2]:
# %rm -rf /content/Titans_jax

In [3]:
!git clone https://github.com/andrew-veriga/Titans_jax.git
%cd Titans_jax

Cloning into 'Titans_jax'...
remote: Enumerating objects: 253, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 253 (delta 19), reused 27 (delta 11), pack-reused 214 (from 1)
Receiving objects: 100% (253/253), 11.08 MiB | 28.59 MiB/s, done.
Resolving deltas: 100% (149/149), done.
/content/Titans_jax


In [4]:
from kauldron import kd
from kauldron.ktyping import Array

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [5]:
import jax
import jax.numpy as jnp
import optax
from kauldron import kd
import numpy as np
import os
import orbax.checkpoint as ocp

# Gemma imports
from gemma import gm

# Our custom Titans integration
import importlib
import gemma_titans
importlib.reload(gemma_titans)
from gemma_titans import Gemma3_1B_Titans
from titans_ckpts import SkipTitans
import titans_tree_utils

print(f"JAX Backend: {jax.default_backend()}")
print(f"Devices: {jax.devices()}")

# Prevent JAX from allocating 100% of TPU memory instantly
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Limit XLA to 85% of TPU HBM to leave room for overhead
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = ".85"
# Reduce fragmentation and compilation memory spike
os.environ["XLA_FLAGS"] = "--xla_gpu_enable_highest_priority_async_collectives=true --xla_tpu_enable_data_parallel_all_reduce_opt=true --xla_tpu_memory_bound_loop_fusion_limit=1"
os.environ["JAX_COMPILATION_CACHE_DIR"] = "/tmp/jax_cache"


JAX Backend: tpu
Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]


## 3. Load Trained Weights
We didn't save the entire 1B model, just our new memory projections.

In [6]:
import google.colab
import shutil

shutil.unpack_archive('/content/Titans_jax/saved_titans_delta.zip',"./saved_titans_delta")

In [7]:
def load_titans_weights(load_dir: str):
    load_path = os.path.abspath(load_dir)
    checkpointer = ocp.StandardCheckpointer()

    # Restore directly without a template (returns dictionary of arrays)
    titans_params = checkpointer.restore(load_path)
    return titans_params

# Usage:

state = load_titans_weights("./saved_titans_delta")

In [8]:
!ls ./saved_titans_delta


array_metadatas       d		      _METADATA        _sharding
_CHECKPOINT_METADATA  manifest.ocdbt  ocdbt.process_0


In [9]:
gemma_config = gm.nn.Gemma3_1B.config

# Initialize model
model = Gemma3_1B_Titans(
    config=gemma_config,
    dtype=jnp.bfloat16,
    return_last_only=False,
    tokens="batch.input",
)
model.titans_layer_indices = [11] # Add Titans memory only to the middle layer

# Load original Gemma 3 1B IT weights
print("Loading original Gemma weights...")
original_params = gm.ckpts.load_params(gm.ckpts.CheckpointPath.GEMMA3_1B_IT)

# Merge loaded Titans weights into original Gemma params
print("Merging Titans weights...")
import titans_tree_utils
loaded_titans_params = load_titans_weights("./saved_titans_delta")
merged_params = titans_tree_utils.merge_titans_params(original_params, loaded_titans_params)

# Create a proper Kauldron dataset pipeline using TFDS
batch_size = 8
max_length = 256

def format_squad(x):
    # SQuAD structure: context, question, answers (dict with 'text' list)
    answers = x.get('answers', {}).get('text', [])
    answer = answers[0] if len(answers) > 0 else "Unknown"

    # Robustly handle potential bytes from TFDS
    ctx = x["context"].decode('utf-8') if hasattr(x["context"], 'decode') else x["context"]
    q = x["question"].decode('utf-8') if hasattr(x["question"], 'decode') else x["question"]
    ans = answer.decode('utf-8') if hasattr(answer, 'decode') else answer

    # Create 'src' and 'dst' for the Seq2SeqTask
    x['src'] = "Context: " + ctx + "\n\nQuestion: " + q
    x['dst'] = ans
    return x

tokenizer = gm.text.Gemma3Tokenizer()


Loading original Gemma weights...


Merging Titans weights...


## 4. Dialogue Inference (Test-Time Dynamic Memory)
During inference, the model passes a `cache` containing both the standard Attention KV-cache AND the Titans Neural Memory State. The model automatically updates its internal state.

In [10]:
# We use Gemma's built-in Sampler to correctly handle positions, attention masks, and cache updates
sampler = gm.text.Sampler(
    model=model,
    params=merged_params,
    tokenizer=tokenizer,
)

print("Simulation: User enters a turn...")
prompt = "Context: The Titans are powerful mythological entities.\n\nQuestion: Who are the Titans?"
prompt = "Titans - технология запоминания контекста во время ответа LLM"
# The sampler will automatically prefill the cache, update Titans memory, and generate text
output = sampler.sample(prompt, max_new_tokens=20)

print("\n--- Generated Response ---")
print(output)
print("\nMemory state updated automatically in cache.")

Simulation: User enters a turn...

--- Generated Response ---
 பணி naar notflächeвого опыходи, качественноřčeníটেড do not ну усилияடுத்து, não ce

Memory state updated automatically in cache.


## 5. (Bonus) Fast Loading via `jax.eval_shape`
In a new session, to avoid wasting time and memory initializing random weights before loading, you can create a zero-memory structural template using `jax.eval_shape`.

In [14]:
sampler.sample("question: Tell me about Colab. Answer:", max_new_tokens=40)

' n el cualობა convid revenir zread mesmo recentemente জানতে ख्याल λε 말씀não overafterduct最近怎么بعد themνα காணેd nدل conosco por তাকে المشasting encamin dànhそれでも porటర్\u200c P par'